In [ ]:
!git clone https://github.com/kingabzpro/human-eval

Cloning into 'human-eval'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 32 (delta 11), reused 5 (delta 5), pack-reused 8 (from 2)
Receiving objects: 100% (32/32), 55.05 KiB | 5.00 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [ ]:
%cd human-eval

/content/human-eval


In [ ]:
!pip install -e .

Obtaining file:///content/human-eval
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.3 MB/s eta 0:00:00
  Running setup.py develop for human-eval


In [ ]:
!evaluate_functional_correctness data/example_samples.jsonl --problem_file=data/example_problem.jsonl

Reading samples...
6it [00:00, 709.94it/s]
Running test suites...
100% 6/6 [00:03<00:00,  1.97it/s]
Writing results to data/example_samples.jsonl_results.jsonl...
100% 6/6 [00:00<00:00, 13231.24it/s]
{'pass@1': np.float64(0.4999999999999999)}


In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define where you want to save outputs in Drive
drive_folder = '/content/drive/MyDrive/human_eval_results'
!mkdir -p {drive_folder}

# 2️⃣ Clone the repo
!git clone https://github.com/kingabzpro/human-eval
%cd human-eval

# 3️⃣ Install the package
!pip install -e .

# 4️⃣ (Optional) Install extra dependencies if any
# !pip install -r requirements.txt

# 5️⃣ Run the evaluation
import os

# Paths to data files
sample_file = "data/example_samples.jsonl"
problem_file = "data/example_problem.jsonl"

# Output file path in Drive
output_file = os.path.join(drive_folder, "evaluation_results.jsonl")

# Run the evaluation command
!evaluate_functional_correctness {sample_file} --problem_file={problem_file} --output_file={output_file}

# 6️⃣ Copy any other generated artifacts to Drive
# For example, saving generated code, test cases, logs
!cp -r data {drive_folder}/data
!cp -r human_eval {drive_folder}/human_eval_repo  # if you want to save the repo

print(f"All results saved to: {drive_folder}")


Mounted at /content/drive
Cloning into 'human-eval'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 32 (delta 11), reused 5 (delta 5), pack-reused 8 (from 2)
Receiving objects: 100% (32/32), 55.05 KiB | 18.35 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/human-eval/human-eval
Obtaining file:///content/human-eval/human-eval
  Preparing metadata (setup.py) ... done
  Attempting uninstall: human-eval
    Found existing installation: human-eval 1.0
    Uninstalling human-eval-1.0:
      Successfully uninstalled human-eval-1.0
  Running setup.py develop for human-eval
Reading samples...
6it [00:00, 659.57it/s]
Running test suites...
100% 6/6 [00:03<00:00,  1.93it/s]
Writing results to data/example_samples.jsonl_results.jsonl...
100% 6/6 [00:00<00:00, 15373.14it/s]
{'pass@1': np.float64(0.4999999999999999)}
ERROR: Could not consume arg: --output_file=/content/drive/MyDrive/hum

In [ ]:
%%capture
%pip install evaluate
%pip install -e .

In [ ]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/human_eval_results'
os.makedirs(drive_folder, exist_ok=True)

# 2️⃣ Install packages

!pip install datasets evaluate tqdm openai google-generativeai tabulate pytest

# 3️⃣ Imports
from datasets import load_dataset
from evaluate import load
from tqdm import tqdm
import json
import re
import os
import openai
import google.generativeai as genai
from openai import OpenAI

# 4️⃣ Set API keys
os.environ["OPENAI_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""

# Initialize clients
gpt = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
GPT_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# 5️⃣ Load HumanEval dataset and code evaluation metric
human_eval = load_dataset("openai_humaneval")['test']
code_eval_metric = load("code_eval")

# 6️⃣ Define CoT / SCoT prompting functions
def chain_of_thought(problem_text):
    return f"""
You are a skilled Python developer.
{problem_text}
Think step by step about the logic before writing the code.
Then write the final, correct function implementation.
"""

def stepwise_chain_of_thought(problem_text):
    return f"""
You are to solve the following problem step-by-step.

1. Break down the logic clearly in small numbered steps.
2. Then write the Python function.
3. Double-check for correctness and edge cases.

Problem:
{problem_text}
"""

prompt_strategies = [("CoT", chain_of_thought), ("SCoT", stepwise_chain_of_thought)]

# 7️⃣ Utility function to extract clean code
def extract_code(text):
    if not text:
        return ""
    text = re.sub(r"```(?:python)?\n", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()

# 8️⃣ Gemini generation function
def generate_code_gemini(prompt):
    model = genai.GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(prompt)
    if hasattr(response, "text") and response.text:
        return extract_code(response.text)
    elif hasattr(response, "candidates") and response.candidates:
        return extract_code(response.candidates[0].content.parts[0].text)
    else:
        return ""

# 9️⃣ Safe pass@1 checker
def check_pass(problem, code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        exec(problem["test"], {}, local_vars)
        return 1
    except Exception:
        return 0

# 10️⃣ Select 10 problems
num_problems = 10
selected_problems = human_eval.select(range(num_problems))

# 11️⃣ Generate code for all models and strategies
generated_results = []

for sample in tqdm(selected_problems):
    problem_id = sample['task_id']
    test_cases = sample['test']

    for strat_name, strat_fn in prompt_strategies:
        prompt = strat_fn(sample['prompt'])

        # ----- GPT-4 -----
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0
        )
        gpt_code = extract_code(response.choices[0].message.content)
        generated_results.append({
            "problem_id": problem_id,
            "model": "GPT-4",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gpt_code,
            "test_cases": test_cases
        })

        # ----- Gemini -----
        gem_code = generate_code_gemini(prompt)
        generated_results.append({
            "problem_id": problem_id,
            "model": "Gemini",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gem_code,
            "test_cases": test_cases
        })

# 12️⃣ Save generated code, prompts, and test cases
generated_jsonl_path = os.path.join(drive_folder, "generated_code_prompts.jsonl")
with open(generated_jsonl_path, "w") as f:
    for entry in generated_results:
        f.write(json.dumps(entry) + "\n")
print(f"Generated code & prompts saved to: {generated_jsonl_path}")

# 13️⃣ Evaluate functional correctness (pass@1)
evaluation_results = []
for entry in tqdm(generated_results):
    score = code_eval_metric.compute(predictions=[entry['generated_code']], references=[entry['test_cases']])
    evaluation_results.append({
        "problem_id": entry["problem_id"],
        "model": entry["model"],
        "strategy": entry["strategy"],
        "score": score
    })

# Save evaluation results
eval_jsonl_path = os.path.join(drive_folder, "evaluation_results.jsonl")
with open(eval_jsonl_path, "w") as f:
    for entry in evaluation_results:
        f.write(json.dumps(entry) + "\n")
print(f"Evaluation results saved to: {eval_jsonl_path}")

# 14️⃣ Debugging & iterative improvement
debugging_results = []

failure_entries = [e for e in evaluation_results if e['score']['pass@1']==0][:2]

for fail in failure_entries:
    orig_entry = next(e for e in generated_results if e['problem_id']==fail['problem_id'] and e['model']==fail['model'] and e['strategy']==fail['strategy'])
    improved_prompt = orig_entry['prompt'] + "\n# Debug: Ensure all edge cases are handled."

    if fail['model'] == "GPT-4":
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": improved_prompt}],
            max_tokens=512,
            temperature=0
        )
        improved_code = extract_code(response.choices[0].message.content)
    else:  # Gemini
        improved_code = generate_code_gemini(improved_prompt)

    improved_score = code_eval_metric.compute(predictions=[improved_code], references=[orig_entry['test_cases']])

    debugging_results.append({
        "problem_id": fail['problem_id'],
        "model": fail['model'],
        "strategy": fail['strategy'],
        "original_code": orig_entry['generated_code'],
        "original_score": fail['score'],
        "improved_prompt": improved_prompt,
        "improved_code": improved_code,
        "improved_score": improved_score
    })

# Save debugging results
debug_jsonl_path = os.path.join(drive_folder, "debugging_results.jsonl")
with open(debug_jsonl_path, "w") as f:
    for entry in debugging_results:
        f.write(json.dumps(entry) + "\n")
print(f"Debugging results saved to: {debug_jsonl_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 10/10 [09:22<00:00, 56.25s/it]


Generated code & prompts saved to: /content/drive/MyDrive/human_eval_results/generated_code_prompts.jsonl


  0%|          | 0/40 [00:00<?, ?it/s]


ValueError: Got a string but expected a list instead: 'To determine if any two numbers in a list are closer to each other than a given threshold, we can follow these steps:

1. **Sort the List**: By sorting the list of numbers, we can ensure that we only need to compare each number with its immediate neighbor. This is because if two numbers are close together in value, they will be adjacent in a sorted list.

2. **Compare Adjacent Elements**: After sorting, we can iterate through the list and compare each number with the next one. If the difference between any two adjacent numbers is less than the threshold, we can return `True`.

3. **Return False if No Close Elements Found**: If we finish checking all adjacent pairs without finding any that are closer than the threshold, we return `False`.

Now, let's implement this logic in the function:

from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are an

In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/human_eval_results'
os.makedirs(drive_folder, exist_ok=True)

# 2️⃣ Install packages
!pip install datasets evaluate tqdm openai google-generativeai tabulate pytest

# 3️⃣ Imports
from datasets import load_dataset
from evaluate import load
from tqdm import tqdm
import json
import re
import os
import openai
import google.generativeai as genai
from openai import OpenAI

# 4️⃣ Set API keys
os.environ["OPENAI_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""

# Initialize clients
gpt = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
GPT_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# 5️⃣ Load HumanEval dataset and code evaluation metric
human_eval = load_dataset("openai_humaneval")['test']
code_eval_metric = load("code_eval")

# 6️⃣ Define CoT / SCoT prompting functions
def chain_of_thought(problem_text):
    return f"""
You are a skilled Python developer.
{problem_text}
Think step by step about the logic before writing the code.
Then write the final, correct function implementation.
"""

def stepwise_chain_of_thought(problem_text):
    return f"""
You are to solve the following problem step-by-step.

1. Break down the logic clearly in small numbered steps.
2. Then write the Python function.
3. Double-check for correctness and edge cases.

Problem:
{problem_text}
"""

prompt_strategies = [("CoT", chain_of_thought), ("SCoT", stepwise_chain_of_thought)]

# 7️⃣ Utility function to extract clean code
def extract_code(text):
    if not text:
        return ""
    text = re.sub(r"```(?:python)?\n", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()

# 8️⃣ Gemini generation function
def generate_code_gemini(prompt):
    model = genai.GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(prompt)
    if hasattr(response, "text") and response.text:
        return extract_code(response.text)
    elif hasattr(response, "candidates") and response.candidates:
        return extract_code(response.candidates[0].content.parts[0].text)
    else:
        return ""

# 9️⃣ Safe pass@1 checker
def check_pass(problem, code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        exec(problem["test"], {}, local_vars)
        return 1
    except Exception:
        return 0

# 10️⃣ Select 10 problems
num_problems = 10
selected_problems = human_eval.select(range(num_problems))

# 11️⃣ Generate code for all models and strategies
generated_results = []

for sample in tqdm(selected_problems):
    problem_id = sample['task_id']

    for strat_name, strat_fn in prompt_strategies:
        prompt = strat_fn(sample['prompt'])

        # ----- GPT-4 -----
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0
        )
        gpt_code = extract_code(response.choices[0].message.content)
        generated_results.append({
            "problem_id": problem_id,
            "model": "GPT-4",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gpt_code,
            "test_cases": sample["test"]
        })

        # ----- Gemini -----
        gem_code = generate_code_gemini(prompt)
        generated_results.append({
            "problem_id": problem_id,
            "model": "Gemini",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gem_code,
            "test_cases": sample["test"]
        })

# 12️⃣ Save generated code, prompts, and test cases
generated_jsonl_path = os.path.join(drive_folder, "generated_code_prompts.jsonl")
with open(generated_jsonl_path, "w") as f:
    for entry in generated_results:
        f.write(json.dumps(entry) + "\n")
print(f"Generated code & prompts saved to: {generated_jsonl_path}")

# 13️⃣ Evaluate functional correctness (pass@1)
evaluation_results = []

for entry in tqdm(generated_results):
    try:
        # Use full problem object as reference for code_eval
        problem = next(p for p in selected_problems if p['task_id'] == entry['problem_id'])
        score = code_eval_metric.compute(predictions=[entry['generated_code']], references=[problem])
    except Exception as e:
        score = {"pass@1": 0, "error": str(e)}

    evaluation_results.append({
        "problem_id": entry["problem_id"],
        "model": entry["model"],
        "strategy": entry["strategy"],
        "score": score
    })

# Save evaluation results
eval_jsonl_path = os.path.join(drive_folder, "evaluation_results.jsonl")
with open(eval_jsonl_path, "w") as f:
    for entry in evaluation_results:
        f.write(json.dumps(entry) + "\n")
print(f"Evaluation results saved to: {eval_jsonl_path}")

# 14️⃣ Debugging & iterative improvement
debugging_results = []

failure_entries = [e for e in evaluation_results if e['score'].get('pass@1', 0)==0][:2]

for fail in failure_entries:
    orig_entry = next(e for e in generated_results if e['problem_id']==fail['problem_id'] and e['model']==fail['model'] and e['strategy']==fail['strategy'])
    improved_prompt = orig_entry['prompt'] + "\n# Debug: Ensure all edge cases are handled."

    if fail['model'] == "GPT-4":
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": improved_prompt}],
            max_tokens=512,
            temperature=0
        )
        improved_code = extract_code(response.choices[0].message.content)
    else:  # Gemini
        improved_code = generate_code_gemini(improved_prompt)

    try:
        improved_score = code_eval_metric.compute(predictions=[improved_code], references=[orig_entry["test_cases"]])
    except:
        improved_score = {"pass@1": 0}

    debugging_results.append({
        "problem_id": fail['problem_id'],
        "model": fail['model'],
        "strategy": fail['strategy'],
        "original_code": orig_entry['generated_code'],
        "original_score": fail['score'],
        "improved_prompt": improved_prompt,
        "improved_code": improved_code,
        "improved_score": improved_score
    })

# Save debugging results
debug_jsonl_path = os.path.join(drive_folder, "debugging_results.jsonl")
with open(debug_jsonl_path, "w") as f:
    for entry in debugging_results:
        f.write(json.dumps(entry) + "\n")
print(f"Debugging results saved to: {debug_jsonl_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 10/10 [09:42<00:00, 58.24s/it]


Generated code & prompts saved to: /content/drive/MyDrive/human_eval_results/generated_code_prompts.jsonl


100%|██████████| 40/40 [00:01<00:00, 38.82it/s]


Evaluation results saved to: /content/drive/MyDrive/human_eval_results/evaluation_results.jsonl
Debugging results saved to: /content/drive/MyDrive/human_eval_results/debugging_results.jsonl


In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/human_eval_results'
os.makedirs(drive_folder, exist_ok=True)

# 2️⃣ Install packages
!pip install datasets evaluate tqdm openai google-generativeai tabulate pytest

# 3️⃣ Imports
from datasets import load_dataset
from evaluate import load
from tqdm import tqdm
import json
import re
import os
import openai
import google.generativeai as genai
from openai import OpenAI

# 4️⃣ Set API keys
os.environ["OPENAI_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""

# Initialize clients
gpt = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
GPT_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# 5️⃣ Load HumanEval dataset and code evaluation metric
human_eval = load_dataset("openai_humaneval")['test']
code_eval_metric = load("code_eval")

# 6️⃣ Define CoT / SCoT prompting functions
def chain_of_thought(problem_text):
    return f"""
You are a skilled Python developer.
{problem_text}
Think step by step about the logic before writing the code.
Then write the final, correct function implementation.
Return the code in a function named as get_solution
Mark the start and end of get_solution with start and end keywords
"""

def stepwise_chain_of_thought(problem_text):
    return f"""
You are to solve the following problem step-by-step.

1. Break down the logic clearly in small numbered steps.
2. Then write the Python function.
3. Double-check for correctness and edge cases.
4. Return the code in a function named as get_solution
5. Mark the start and end of get_solution with start and end keywords

Problem:
{problem_text}
"""

prompt_strategies = [("CoT", chain_of_thought), ("SCoT", stepwise_chain_of_thought)]

# 7️⃣ Utility function to extract clean code
def extract_code(text):
    if not text:
        return ""
    text = re.sub(r"```(?:python)?\n", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()

# 8️⃣ Gemini generation function
def generate_code_gemini(prompt):
    model = genai.GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(prompt)
    if hasattr(response, "text") and response.text:
        return extract_code(response.text)
    elif hasattr(response, "candidates") and response.candidates:
        return extract_code(response.candidates[0].content.parts[0].text)
    else:
        return ""

# 9️⃣ Safe pass@1 checker
def check_pass(problem, code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        exec(problem["test"], {}, local_vars)
        return 1
    except Exception:
        return 0

# 10️⃣ Select 10 problems
num_problems = 10
selected_problems = human_eval.select(range(num_problems))

# 11️⃣ Generate code for all models and strategies
generated_results = []

for sample in tqdm(selected_problems):
    problem_id = sample['task_id']

    for strat_name, strat_fn in prompt_strategies:
        prompt = strat_fn(sample['prompt'])

        # ----- GPT-4 -----
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0
        )
        gpt_code = extract_code(response.choices[0].message.content)
        generated_results.append({
            "problem_id": problem_id,
            "model": "GPT-4",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gpt_code,
            "test_cases": sample["test"]
        })

        # ----- Gemini -----
        gem_code = generate_code_gemini(prompt)
        generated_results.append({
            "problem_id": problem_id,
            "model": "Gemini",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gem_code,
            "test_cases": sample["test"]
        })

# 12️⃣ Save generated code, prompts, and test cases
generated_jsonl_path = os.path.join(drive_folder, "generated_code_prompts.jsonl")
with open(generated_jsonl_path, "w") as f:
    for entry in generated_results:
        f.write(json.dumps(entry) + "\n")
print(f"Generated code & prompts saved to: {generated_jsonl_path}")

# Helper: Extract get_solution code (brute-force, no regex)
def extract_get_solution_code(text):
    if not text:
        return ""
    lines = text.splitlines()
    start_idx, end_idx = None, None

    for i, line in enumerate(lines):
        if 'start' in line.strip().lower() and start_idx is None:
            start_idx = i + 1
        elif 'end' in line.strip().lower() and start_idx is not None:
            end_idx = i
            break

    if start_idx is not None and end_idx is not None and start_idx < end_idx:
        return "\n".join(lines[start_idx:end_idx]).strip()
    return text.strip()


# 13️⃣ Evaluate functional correctness (pass@1)
evaluation_results = []

for entry in tqdm(generated_results):
    try:
        # Extract only the actual function code between 'start' and 'end'
        clean_code = extract_get_solution_code(entry["generated_code"])

        # Get problem object
        problem = next(p for p in selected_problems if p["task_id"] == entry["problem_id"])

        # Evaluate
        score = code_eval_metric.compute(predictions=[clean_code], references=[problem])
    except Exception as e:
        score = {"pass@1": 0, "error": str(e)}

    evaluation_results.append({
        "problem_id": entry["problem_id"],
        "model": entry["model"],
        "strategy": entry["strategy"],
        "generated_code": clean_code,
        "score": score
    })

# Save evaluation results
eval_jsonl_path = os.path.join(drive_folder, "evaluation_results.jsonl")
with open(eval_jsonl_path, "w") as f:
    for entry in evaluation_results:
        f.write(json.dumps(entry) + "\n")
print(f"✅ Evaluation results saved to: {eval_jsonl_path}")

# Identify failed cases for debugging (Part 2)
failed_cases = [
    entry for entry in evaluation_results
    if entry["score"].get("pass@1", 0) == 0
]
print(f"🔍 Found {len(failed_cases)} failed cases for debugging.")


def generate_code_with_model(model_name, prompt):
    if model_name.lower().startswith("gpt"):
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0
        )
        return response.choices[0].message.content
    else:
        return generate_code_gemini(prompt)

# 🔁 Debugging & Iterative Improvement
debug_results = []

for fail_case in failed_cases:
    try:
        problem_text = next(p['prompt'] for p in selected_problems if p["task_id"] == fail_case["problem_id"])
        improved_prompt = f"""
The following solution failed test cases. Debug and fix the errors.
Keep the same function name get_solution.
Provide only the corrected function code.
Mark the start and end of get_solution with start and end keywords.

Failed code:
{fail_case['generated_code']}

Problem:
{problem_text}
"""

        improved_output = generate_code_with_model(
            model_name=fail_case["model"],
            prompt=improved_prompt
        )

        # Extract fixed function only
        fixed_code = extract_get_solution_code(improved_output)

        problem = next(p for p in selected_problems if p["task_id"] == fail_case["problem_id"])
        score = code_eval_metric.compute(predictions=[fixed_code], references=[problem])

    except Exception as e:
        score = {"pass@1": 0, "error": str(e)}

    debug_results.append({
        "problem_id": fail_case["problem_id"],
        "model": fail_case["model"],
        "original_strategy": fail_case["strategy"],
        "fixed_score": score,
    })

# Save debugging results
debug_jsonl_path = os.path.join(drive_folder, "debugging_results.jsonl")
with open(debug_jsonl_path, "w") as f:
    for entry in debug_results:
        f.write(json.dumps(entry) + "\n")
print(f"✅ Debugging results saved to: {debug_jsonl_path}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 10/10 [07:39<00:00, 45.92s/it]


Generated code & prompts saved to: /content/drive/MyDrive/human_eval_results/generated_code_prompts.jsonl


100%|██████████| 40/40 [00:00<00:00, 1739.38it/s]


✅ Evaluation results saved to: /content/drive/MyDrive/human_eval_results/evaluation_results.jsonl
🔍 Found 40 failed cases for debugging.


KeyboardInterrupt: 

In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/human_eval_results'
os.makedirs(drive_folder, exist_ok=True)

# 2️⃣ Install packages
!pip install datasets evaluate tqdm openai google-generativeai tabulate pytest

# 3️⃣ Imports
from datasets import load_dataset
from evaluate import load
from tqdm import tqdm
import json
import re
import os
import time
import openai
import google.generativeai as genai
from openai import OpenAI

# Extra imports for per-case comparisons (kept from your previous version if present)
import ast
import math
import itertools
import collections
import traceback

# 4️⃣ Set API keys
os.environ["OPENAI_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""

# Initialize clients
gpt = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
GPT_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# Control flags
USE_GEMINI = True          # Set False to skip Gemini quickly during debugging
GEMINI_TIMEOUT_S = 45      # Hard deadline per request
GEMINI_MAX_RETRIES = 3     # Bounded retries
GEMINI_BACKOFF_BASE = 2.0  # Exponential backoff

# 5️⃣ Load HumanEval dataset and code evaluation metric
human_eval = load_dataset("openai_humaneval")['test']
code_eval_metric = load("code_eval")

# 6️⃣ Define CoT / SCoT prompting functions
def chain_of_thought(problem_text):
    return f"""
You are a skilled Python developer.
{problem_text}
Think step by step about the logic before writing the code.
Then write the final, correct function implementation.
Return the code in a function named as get_solution
Mark the start and end of get_solution with start and end keywords
"""

def stepwise_chain_of_thought(problem_text):
    return f"""
You are to solve the following problem step-by-step.

1. Break down the logic clearly in small numbered steps.
2. Then write the Python function.
3. Double-check for correctness and edge cases.
4. Return the code in a function named as get_solution
5. Mark the start and end of get_solution with start and end keywords

Problem:
{problem_text}
"""

prompt_strategies = [("CoT", chain_of_thought), ("SCoT", stepwise_chain_of_thought)]

# 7️⃣ Utility function to extract clean code
def extract_code(text):
    if not text:
        return ""
    text = re.sub(r"```(?:python)?\n", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()

# Helper: Extract get_solution code (brute-force, no regex)
def extract_get_solution_code(text):
    if not text:
        return ""
    lines = text.splitlines()
    start_idx, end_idx = None, None
    for i, line in enumerate(lines):
        ls = line.strip().lower()
        if 'start' in ls and start_idx is None:
            start_idx = i + 1
            continue
        if 'end' in ls and start_idx is not None:
            end_idx = i
            break
    if start_idx is not None and end_idx is not None and start_idx < end_idx:
        return "\n".join(lines[start_idx:end_idx]).strip()
    return text.strip()

# 8️⃣ Gemini generation function WITH TIMEOUT + RETRIES + SKIP ON FAILURE
def generate_code_gemini(prompt, timeout_s=GEMINI_TIMEOUT_S, max_retries=GEMINI_MAX_RETRIES):
    if not USE_GEMINI:
        return ""
    last_err = None
    for attempt in range(max_retries):
        try:
            model = genai.GenerativeModel(GEMINI_MODEL)
            # request_options accepts "timeout" for per-call deadline in google.generativeai
            # (other options are not supported here)
            response = model.generate_content(
                prompt,
                request_options={"timeout": float(timeout_s)}
            )
            # Prefer text when available
            if hasattr(response, "text") and response.text:
                return extract_code(response.text)
            if hasattr(response, "candidates") and response.candidates:
                part = response.candidates[0].content.parts[0]
                text = getattr(part, "text", None) or getattr(part, "raw_text", "")
                return extract_code(text or "")
            return ""
        except Exception as e:
            last_err = e
            # Exponential backoff before retry
            sleep_s = (GEMINI_BACKOFF_BASE ** attempt)
            time.sleep(min(8.0, sleep_s))
    # After retries, return empty string to let the pipeline continue
    print(f"[Gemini] Skipping after retries due to error: {type(last_err).__name__}: {last_err}")
    return ""

# 9️⃣ Safe pass@1 checker (kept)
def check_pass(problem, code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        exec(problem["test"], {}, local_vars)
        return 1
    except Exception:
        return 0

# 10️⃣ Select 10 problems
num_problems = 10
selected_problems = human_eval.select(range(num_problems))

# Optional: helpers for per-case comparisons (keep your previous ones if already added)
SAFE_EVAL_GLOBALS = {
    "math": math,
    "itertools": itertools,
    "collections": collections,
    "Counter": collections.Counter,
    "deque": collections.deque,
    "set": set,
    "list": list,
    "tuple": tuple,
    "dict": dict,
    "len": len,
    "sum": sum,
    "max": max,
    "min": min,
    "sorted": sorted,
    "abs": abs,
    "all": all,
    "any": any,
    "enumerate": enumerate,
    "range": range,
}

def get_entry_point(problem):
    ep = problem.get("entry_point", None)
    if ep:
        return ep
    try:
        for line in problem["prompt"].splitlines():
            ls = line.strip()
            if ls.startswith("def ") and "(" in ls:
                return ls.split("def ", 1)[1].split("(", 1)[0].strip()
    except Exception:
        pass
    return None

def node_src(source, node):
    try:
        return ast.get_source_segment(source, node)
    except Exception:
        return None

def try_eval_expr(expr_src):
    if not expr_src:
        return None, "no_expr_src"
    try:
        return eval(compile(ast.parse(expr_src, mode="eval"), "<expr>", "eval"), SAFE_EVAL_GLOBALS, {}), None
    except Exception as e:
        return None, f"eval_error: {e}"

def extract_assert_cases(test_code, entry_point_name):
    results = []
    try:
        tree = ast.parse(test_code)
        for node in ast.walk(tree):
            if not isinstance(node, ast.Assert):
                continue
            test = node.test
            if not isinstance(test, ast.Compare):
                continue
            if not isinstance(test.left, ast.Call):
                continue
            call = test.left
            func = call.func
            if not (isinstance(func, ast.Name) and func.id == entry_point_name):
                continue
            if len(test.comparators) != 1:
                continue
            args_vals, kwargs_vals = [], {}
            parse_error = None
            for a in call.args:
                a_src = node_src(test_code, a)
                val, err = try_eval_expr(a_src)
                if err:
                    parse_error = parse_error or f"arg_eval_error: {err}"
                args_vals.append(val)
            for kw in call.keywords or []:
                k = kw.arg
                v_src = node_src(test_code, kw.value)
                val, err = try_eval_expr(v_src)
                if err:
                    parse_error = parse_error or f"kwarg_eval_error: {err}"
                kwargs_vals[k] = val
            expected_src = node_src(test_code, test.comparators[0])
            expected_val, exp_err = try_eval_expr(expected_src)
            if exp_err:
                parse_error = parse_error or f"expected_eval_error: {exp_err}"
            raw = node_src(test_code, node) or ""
            results.append({
                "args": args_vals,
                "kwargs": kwargs_vals,
                "expected": expected_val,
                "expected_src": expected_src,
                "raw_assert": raw,
                "parse_error": parse_error
            })
    except Exception as e:
        results.append({
            "args": None,
            "kwargs": None,
            "expected": None,
            "expected_src": None,
            "raw_assert": None,
            "parse_error": f"ast_parse_error: {e}"
        })
    return results

def run_per_case_comparisons(problem, clean_code_string):
    entry_point = get_entry_point(problem)
    test_code = problem["test"]
    env = {}
    errors = {}
    try:
        exec(clean_code_string, env, env)
    except Exception as e:
        errors["exec_error"] = f"{type(e).__name__}: {e}"
        return {
            "entry_point": entry_point,
            "overall_passed": 0,
            "exec_error": errors.get("exec_error"),
            "cases": [],
        }
    if "get_solution" not in env or not callable(env["get_solution"]):
        return {
            "entry_point": entry_point,
            "overall_passed": 0,
            "exec_error": "get_solution not found or not callable",
            "cases": [],
        }
    overall_passed = 0
    try:
        run_env = dict(env)
        if entry_point and entry_point != "get_solution":
            run_env[entry_point] = run_env["get_solution"]
        exec(test_code, run_env, run_env)
        overall_passed = 1
    except Exception:
        overall_passed = 0
    cases = []
    assert_cases = extract_assert_cases(test_code, entry_point) if entry_point else []
    for idx, c in enumerate(assert_cases):
        if c.get("args") is None and c.get("kwargs") is None:
            cases.append({
                "index": idx,
                "inputs": None,
                "expected_output": c.get("expected"),
                "expected_src": c.get("expected_src"),
                "actual_output": None,
                "passed": False,
                "error": c.get("parse_error"),
                "raw_assert": c.get("raw_assert"),
            })
            continue
        try:
            actual = env["get_solution"](*(c["args"] or []), **(c["kwargs"] or {}))
            passed = (c["expected"] == actual)
            cases.append({
                "index": idx,
                "inputs": {"args": c["args"], "kwargs": c["kwargs"]},
                "expected_output": c["expected"],
                "expected_src": c.get("expected_src"),
                "actual_output": actual,
                "passed": passed,
                "error": None,
                "raw_assert": c.get("raw_assert"),
            })
        except Exception as e:
            cases.append({
                "index": idx,
                "inputs": {"args": c["args"], "kwargs": c["kwargs"]},
                "expected_output": c["expected"],
                "expected_src": c.get("expected_src"),
                "actual_output": None,
                "passed": False,
                "error": f"{type(e).__name__}: {e}",
                "raw_assert": c.get("raw_assert"),
            })
    return {
        "entry_point": entry_point,
        "overall_passed": overall_passed,
        "exec_error": errors.get("exec_error"),
        "cases": cases,
    }

# 11️⃣ Generate code for all models and strategies
generated_results = []
per_case_comparisons = []

for sample in tqdm(selected_problems, desc="Problems"):
    problem_id = sample['task_id']

    for strat_name, strat_fn in prompt_strategies:
        prompt = strat_fn(sample['prompt'])

        # ----- GPT-4 -----
        response = gpt.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0
        )
        gpt_code_full = extract_code(response.choices[0].message.content)
        gpt_code_clean = extract_get_solution_code(gpt_code_full)

        generated_results.append({
            "problem_id": problem_id,
            "model": "GPT-4",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gpt_code_full,
            "clean_code": gpt_code_clean,
            "test_cases": sample["test"]
        })

        comp_gpt = run_per_case_comparisons(sample, gpt_code_clean)
        per_case_comparisons.append({
            "problem_id": problem_id,
            "model": "GPT-4",
            "strategy": strat_name,
            "comparison": comp_gpt
        })

        # ----- Gemini (timeout+retries+skip) -----
        gem_code_full = generate_code_gemini(prompt)
        gem_code_clean = extract_get_solution_code(gem_code_full) if gem_code_full else ""

        generated_results.append({
            "problem_id": problem_id,
            "model": "Gemini",
            "strategy": strat_name,
            "prompt": prompt,
            "generated_code": gem_code_full,
            "clean_code": gem_code_clean,
            "test_cases": sample["test"]
        })

        if gem_code_clean:
            comp_gem = run_per_case_comparisons(sample, gem_code_clean)
        else:
            comp_gem = {
                "entry_point": get_entry_point(sample),
                "overall_passed": 0,
                "exec_error": "gemini_generation_empty_or_timeout",
                "cases": [],
            }
        per_case_comparisons.append({
            "problem_id": problem_id,
            "model": "Gemini",
            "strategy": strat_name,
            "comparison": comp_gem
        })

# 12️⃣ Save generated code, prompts, and test cases
generated_jsonl_path = os.path.join(drive_folder, "generated_code_prompts.jsonl")
with open(generated_jsonl_path, "w") as f:
    for entry in generated_results:
        f.write(json.dumps(entry) + "\n")
print(f"Generated code & prompts saved to: {generated_jsonl_path}")

# 13️⃣ Evaluate functional correctness (pass@1)
evaluation_results = []
for entry in tqdm(generated_results, desc="Metric eval"):
    problem = next(p for p in selected_problems if p["task_id"] == entry["problem_id"])
    clean_code = entry["clean_code"]
    try:
        score = code_eval_metric.compute(predictions=[clean_code], references=[problem])
    except Exception as e:
        score = {"pass@1": 0, "error": str(e)}
    evaluation_results.append({
        "problem_id": entry["problem_id"],
        "model": entry["model"],
        "strategy": entry["strategy"],
        "generated_code": clean_code,
        "score": score
    })

# Save evaluation results
eval_jsonl_path = os.path.join(drive_folder, "evaluation_results.jsonl")
with open(eval_jsonl_path, "w") as f:
    for entry in evaluation_results:
        f.write(json.dumps(entry) + "\n")
print(f"✅ Evaluation results saved to: {eval_jsonl_path}")

# 14️⃣ Save per-case comparisons
per_case_jsonl_path = os.path.join(drive_folder, "per_case_comparisons.jsonl")
with open(per_case_jsonl_path, "w") as f:
    for entry in per_case_comparisons:
        f.write(json.dumps(entry) + "\n")
print(f"✅ Per-case comparisons saved to: {per_case_jsonl_path}")

# 15️⃣ Identify failed cases for debugging (Part 2)
failed_cases = [
    entry for entry in evaluation_results
    if entry["score"].get("pass@1", 0) == 0
]
print(f"🔍 Found {len(failed_cases)} failed cases for debugging.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Problems:   0%|          | 0/10 [00:00<?, ?it/s]

[Gemini] Skipping after retries due to error: ReadTimeout: HTTPConnectionPool(host='localhost', port=36807): Read timed out. (read timeout=45.0)


Problems:  10%|█         | 1/10 [05:07<46:07, 307.49s/it]

[Gemini] Skipping after retries due to error: ReadTimeout: HTTPConnectionPool(host='localhost', port=36807): Read timed out. (read timeout=45.0)
[Gemini] Skipping after retries due to error: ReadTimeout: HTTPConnectionPool(host='localhost', port=36807): Read timed out. (read timeout=45.0)


...

[Gemini] Skipping after retries due to error: ReadTimeout: HTTPConnectionPool(host='localhost', port=36807): Read timed out. (read timeout=45.0)
[Gemini] Skipping after retries due to error: ReadTimeout: HTTPConnectionPool(host='localhost', port=36807): Read timed out. (read timeout=45.0)


Problems: 100%|██████████| 10/10 [50:40<00:00, 304.07s/it]


[Gemini] Skipping after retries due to error: ReadTimeout: HTTPConnectionPool(host='localhost', port=36807): Read timed out. (read timeout=45.0)
Generated code & prompts saved to: /content/drive/MyDrive/human_eval_results/generated_code_prompts.jsonl


Metric eval: 100%|██████████| 40/40 [00:00<00:00, 1422.15it/s]

✅ Evaluation results saved to: /content/drive/MyDrive/human_eval_results/evaluation_results.jsonl
✅ Per-case comparisons saved to: /content/drive/MyDrive/human_eval_results/per_case_comparisons.jsonl
🔍 Found 40 failed cases for debugging.
